In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from feature_engine.encoding import CountFrequencyEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import shap
import mlflow.sklearn
import mlflow

import config

In [ ]:
# define Mlflow env. settings
mlflow.set_tracking_uri(uri=config.mlflow_uri)
%env MLFLOW_TRACKING_URI= config.mlflow_db
%env mlflow_tracking_username = config.mlflow_un
%env mlflow_tracking_password = config.mlflow_pw
#mlflow.enable_system_metrics_logging()

In [ ]:
def describe_top_shap_values(shap_values, X, feature_names, top_n=3):
    """
    Generate a textual description of the top contributing SHAP values for each prediction.

    Args:
        shap_values: SHAP values obtained from the explainer
        X: The input data used for prediction
        feature_names: List of feature names
        top_n: Number of top contributing features to include in the description

    Returns:
        A list of textual descriptions for each prediction
    """

    descriptions = []

    for i in range(len(X)):
        row_shap_values = shap_values[i]
        top_indices = sorted(range(len(row_shap_values)), key=lambda k: abs(row_shap_values[k]), reverse=True)[:top_n]

        explanation = f"The top contributing factors were:"
        for idx in top_indices:
            feature_name = feature_names[idx]
            shap_value = row_shap_values[idx]
            feature_value = X.iloc[i, idx]

            if shap_value > 0:
                effect = "increased"
            else:
                effect = "decreased"

            explanation += f"\n- {feature_name} ({feature_value}) {effect} the prediction by {abs(shap_value):.2f}."

        descriptions.append(explanation)

    return descriptions


In [3]:
mlflow.set_experiment("T0083 - Audit Review")

with mlflow.start_run(run_name=f"Sklearn-RF Inference", nested=True) as greatgrandparent_run:
    for customer in config.customerList: 
        for auditLog in config.auditList:  
            for FilterType in config.FilterTypeList:
                with mlflow.start_run(run_name=f"Audit Type context-{auditLog}-{customer}", nested=True) as grandparent_run:
                    filepath = Path(f'{config.data_location}FS_audit_type_table_name/{auditLog}/{FilterType}/FS_audit_type_table_name_{auditLog}_{FilterType}_{customer}.parquet')
                    if filepath.is_file()==True:
                        ### Replace code - Add encoding script and load data for inference.
                        df = pd.read_parquet(filepath, engine='pyarrow')
                        #df = df.drop(['T0083','T0083_SEVERITY','T_MLAR','T_MLAR_SEVERITY','MLAR_TYPE'],axis=1) #RK should be already done in FS scripts
                        #df = df.astype(str).replace('nan',np.nan).dropna(how='all',axis=1) #RK should be already done in FS scripts
                        df = df.astype(str)
                        features = df.columns.tolist()
                        categoricalColumns = df.select_dtypes(include=['object']).columns.tolist()
                        encoder = CountFrequencyEncoder(encoding_method='frequency', variables=categoricalColumns, ignore_format=True)
                        dataFrame = encoder.fit_transform(df) 
                        #Update to Anomaly field onced changed
                        X_train, X_test, y_train, y_test = train_test_split(
                            dataFrame[dataFrame.columns.drop(list(dataFrame.filter(like='T0083_SEVERITY')))], 
                            dataFrame.filter(like='T0083_SEVERITY').sum(axis=1), 
                            test_size=0.30, 
                            random_state=1234)
                        ###   
                        
                        # Define model parameters
                        model_name = f"Sklearn-RF_{auditLog}_{FilterType}_{customer}_audit_type_model"
                        model_version = "latest"
                        model_uri = f"models:/{model_name}/{model_version}"
                        rf_model = mlflow.sklearn.load_model(model_uri)
                    
                        # Start an MLflow run
                        with mlflow.start_run(run_name=f"Model Prediction", nested=True) as parent_run:
                            # Predict anomalies and generate histogram
                            predicted = pd.DataFrame(rf_model.predict(X_test.values)) 
                            
                            # Combine the scores with the encoded data and unencoded data 
                            predictedDF = pd.concat([pd.DataFrame(data=predicted,columns=['anomaly']),dataFrame],axis=1)
                            predictedDF = predictedDF[predictedDF['anomaly'] > 0]
                            #predictedDF = predictedDF.iloc[:100,1:]

                            finalDF = pd.concat([pd.DataFrame(data=predicted,columns=['anomaly']),df],axis=1)
                            finalDF = finalDF[finalDF['anomaly'] > 0]
                            #finalDF = finalDF.iloc[:100,1:]

                            #Export Dataset
                            mlflow.log_table(finalDF,"finalDF.json")

                            with mlflow.start_run(run_name=f"Row Explaination", nested=True) as child_run:
                                #Train explaination model
                                #Replace below with load 'saved' explainer
                                explainer = shap.TreeExplainer(rf_model, feature_names=rf_model.feature_names_in_) 
                                shap_values = explainer(predictedDF)
                                feature_names = shap_values.feature_names
                                
                                # Generate descriptions for each prediction
                                explanation = describe_top_shap_values(shap_values.values, predictedDF, feature_names)
                                finalDF['local shap explaination'] = explanation
                                
                                ## plot generation - remove later
                                #shapForcePlot = plt.figure(figsize=(9, 3))
                                #shap.force_plot(explainer.expected_value[0], shap_values[1,0], features=explainer.feature_names, matplotlib=True, show=True) #shap_values.values
                                #mlflow.log_figure(shapForcePlot, f"_{customer}_LocalForcePlot.png")  

                                finalDF = finalDF.loc[:, finalDF.columns.isin(['anomaly','SAMPLE_NUMBER','AUDIT_TIMESTAMP','AUDIT_COUNTER','TRANS_ORDER','AUDIT_TYPE','AUDIT_ACTION','TABLE_NAME','TABLE_KEY','USER_NAME','local shap explaination'])]
                                finalDF.to_parquet(f'{auditLog}_finalDf.parquet')                    else:
                    else:    
                        print("Error - Feature store file not found. Check directory.")

In [4]:
import mlflow
mlflow.end_run()